# Phase 3 - Fine-tune DistilBERT for financial sentiment

Trains two variants on Google Colab (free T4 GPU):

1. **standard** - plain cross-entropy
2. **weighted** - class-weighted cross-entropy, weights from the train split only

Model selection uses **macro F1**, not accuracy. The dataset is 60% `neutral`,
so accuracy rewards a model for ignoring the minority `negative` class - which is
the class a risk classifier exists to catch.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

See `notebooks/README.md` for the full upload/download procedure.

---

### transformers 5.x note

This notebook is written against **transformers 5.15.0**, not 4.x. Arguments used
below that differ from the 4.x tutorials found online:

| 4.x | 5.x (used here) |
|---|---|
| `evaluation_strategy=` | `eval_strategy=` |
| `warmup_ratio=0.1` | `warmup_steps=0.1` (float = ratio of total steps) |
| `Trainer(tokenizer=...)` | `Trainer(processing_class=...)` |
| `compute_loss(self, model, inputs, return_outputs=False)` | must also accept `num_items_in_batch=None` |
| `save_safetensors=` | removed (always safetensors) |
| `no_cuda=` | `use_cpu=` |

The constants below mirror `src/config.py`. **Keep them in sync** if you change
either file.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, "
        "then re-run this cell."
    )
print("device:", torch.cuda.get_device_name(0))

## 2. Install pinned dependencies

Versions are pinned to match the local environment so results are reproducible
there. Colab ships older `transformers` by default; this will restart-prompt if
it upgrades an already-imported package.

In [ ]:
!pip install -q "transformers==5.15.0" "datasets==5.0.1" "accelerate>=1.1.0" "scikit-learn==1.9.0"

import transformers, datasets, sklearn

print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("sklearn     ", sklearn.__version__)

## 3. Upload the data

Upload **three** files from your local repo:

- `data/processed/train.csv`
- `data/processed/val.csv`
- `results/baseline_metrics.json`  (needed for the Phase 2 comparison table)

`test.csv` is deliberately **not** uploaded. It is never touched during training.

In [ ]:
import os

from google.colab import files

# Check the FILESYSTEM, not the return value of files.upload().
# files.upload() reports only what *this* invocation received, so uploading in
# two goes (or re-running after a partial upload) makes a return-value check
# report files missing that are in fact already sitting in /content.
files.upload()

REQUIRED = ["train.csv", "val.csv", "baseline_metrics.json"]
missing = [name for name in REQUIRED if not os.path.exists(name)]

print("present:", [name for name in REQUIRED if os.path.exists(name)])
if missing:
    raise SystemExit(f"Missing {missing} - re-run this cell and upload them.")

if os.path.exists("test.csv"):
    os.remove("test.csv")
    raise SystemExit(
        "test.csv was uploaded and has been deleted. The test split is reserved "
        "for final evaluation only - re-run this cell without it."
    )

print("all required files present")

## 4. Config (mirrors `src/config.py`)

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
NUM_EPOCHS = 4
BATCH_SIZE = 16
RANDOM_SEED = 42
LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_LABELS = len(LABEL_NAMES)

print({"model": MODEL_CHECKPOINT, "max_length": MAX_LENGTH, "lr": LEARNING_RATE,
       "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "seed": RANDOM_SEED})

## 5. Seeds

All RNGs are seeded from `RANDOM_SEED`. **GPU training is still not fully
deterministic** - cuDNN kernel selection and non-deterministic CUDA reduction
order mean two runs with identical seeds can differ slightly. Full determinism
would need `torch.use_deterministic_algorithms(True)` plus
`CUBLAS_WORKSPACE_CONFIG=:4096:8`, at a real speed cost. Expect small run-to-run
variation in the third decimal place.

In [ ]:
import random

import numpy as np
import torch
from transformers import set_seed


def set_all_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


set_all_seeds()
print("seeded with", RANDOM_SEED)

## 6. Load and tokenize

No padding at tokenization time - `DataCollatorWithPadding` pads per batch,
which is faster than padding every sequence to 128 when the median sentence is
about 21 whitespace tokens.

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
print(f"train: {len(train_df):,}   val: {len(val_df):,}")
print(train_df["label_name"].value_counts().to_string())

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


def tokenize_frame(frame):
    dataset = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
    )


train_ds = tokenize_frame(train_df)
val_ds = tokenize_frame(val_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

lengths = [len(x) for x in train_ds["input_ids"]]
print(f"subword tokens - max {max(lengths)}, mean {np.mean(lengths):.1f}")
print(f"truncated at {MAX_LENGTH}: {sum(l >= MAX_LENGTH for l in lengths)} sequences")

## 7. Metrics

`macro_f1` is the model-selection metric.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

LABEL_RANGE = list(range(NUM_LABELS))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    per_f1 = f1_score(labels, preds, average=None, labels=LABEL_RANGE, zero_division=0)
    per_recall = recall_score(labels, preds, average=None, labels=LABEL_RANGE, zero_division=0)

    metrics = {
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", labels=LABEL_RANGE, zero_division=0)),
        "weighted_f1": float(f1_score(labels, preds, average="weighted", labels=LABEL_RANGE, zero_division=0)),
    }
    for i, name in enumerate(LABEL_NAMES):
        metrics[f"f1_{name}"] = float(per_f1[i])
        metrics[f"recall_{name}"] = float(per_recall[i])
    return metrics

## 8. Class weights (train split only)

Computed on train alone. Deriving them from val or the full corpus would leak
the evaluation distribution into training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=train_df["label"].to_numpy(),
).astype(np.float32)

print("computed class weights (balanced, train split only):")
for name, weight in zip(LABEL_NAMES, class_weights):
    count = int((train_df["label_name"] == name).sum())
    print(f"  {name:>8}  n={count:>5}  weight={weight:.4f}")

## 9. Weighted-loss Trainer

`num_items_in_batch` is **new in transformers 5.x**. The override must accept it
or Trainer's internal call raises `TypeError`.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


class WeightedLossTrainer(Trainer):
    """Trainer with class-weighted cross-entropy."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = torch.tensor(class_weights, dtype=logits.dtype, device=logits.device)
        loss = torch.nn.functional.cross_entropy(
            logits.view(-1, NUM_LABELS), labels.view(-1), weight=weights
        )
        inputs["labels"] = labels
        return (loss, outputs) if return_outputs else loss

## 10. Training helper

`load_best_model_at_end=True` with `metric_for_best_model="macro_f1"` restores
the best epoch by macro F1 - **not** the last epoch, and **not** best accuracy.

In [ ]:
def build_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",      # 4.x called this evaluation_strategy
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        weight_decay=0.01,
        warmup_steps=0.1,           # 5.x reads a float as a ratio of total steps
        seed=RANDOM_SEED,
        data_seed=RANDOM_SEED,
        fp16=True,                  # T4 supports fp16; roughly halves training time
        report_to="none",
    )


def train_variant(name, trainer_class, save_dir):
    print(f"\n{'=' * 66}\nTRAINING: {name}\n{'=' * 66}")
    set_all_seeds()

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=NUM_LABELS,
        id2label={i: n for i, n in enumerate(LABEL_NAMES)},
        label2id={n: i for i, n in enumerate(LABEL_NAMES)},
    )

    trainer = trainer_class(
        model=model,
        args=build_args(f"checkpoints_{name}"),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        processing_class=tokenizer,   # 4.x called this tokenizer=
        compute_metrics=compute_metrics,
    )

    trainer.train()
    final = trainer.evaluate()

    print(f"\nbest checkpoint by macro_f1: {trainer.state.best_model_checkpoint}")
    print(f"best macro_f1: {trainer.state.best_metric:.4f}")

    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"saved to {save_dir}/")

    return trainer, final

## 11. Variant (a) - standard cross-entropy

In [ ]:
standard_trainer, standard_final = train_variant(
    "standard", Trainer, "distilbert_standard"
)

## 12. Variant (b) - class-weighted cross-entropy

In [ ]:
weighted_trainer, weighted_final = train_variant(
    "weighted", WeightedLossTrainer, "distilbert_weighted"
)

## 13. Per-epoch history

Saved to `distilbert_history.json` for the results log.

In [ ]:
import json


def epoch_rows(trainer):
    """Eval rows, one per epoch (plus a final row from the post-restore evaluate)."""
    return [
        {k: v for k, v in entry.items() if k != "step"}
        for entry in trainer.state.log_history
        if "eval_macro_f1" in entry
    ]


def train_rows(trainer):
    """Training-loss rows.

    Captured separately because they are the evidence for overfitting: without
    them the history file records only eval metrics, and the train-vs-val loss
    divergence has to be read off the console instead of a committed file.
    """
    return [
        {k: v for k, v in entry.items() if k != "step"}
        for entry in trainer.state.log_history
        if "loss" in entry and "eval_loss" not in entry
    ]


def variant_payload(trainer, final):
    return {
        "epochs": epoch_rows(trainer),
        "train_logs": train_rows(trainer),
        "best_macro_f1": float(trainer.state.best_metric),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "final_eval": {
            k: float(v) for k, v in final.items() if isinstance(v, (int, float))
        },
    }


history = {
    "phase": "3-distilbert",
    "evaluated_on": "val",
    "base_model": MODEL_CHECKPOINT,
    "hyperparameters": {
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "weight_decay": 0.01,
        "warmup_steps_ratio": 0.1,
        "fp16": True,
        "seed": RANDOM_SEED,
    },
    "model_selection": {"metric": "macro_f1", "greater_is_better": True},
    "class_weights": {n: float(w) for n, w in zip(LABEL_NAMES, class_weights)},
    "gpu": torch.cuda.get_device_name(0),
    "determinism_note": (
        "All seeds set from RANDOM_SEED, but GPU training is not fully "
        "deterministic; expect variation in the third decimal place."
    ),
    "variants": {
        "distilbert_standard": variant_payload(standard_trainer, standard_final),
        "distilbert_weighted": variant_payload(weighted_trainer, weighted_final),
    },
}

with open("distilbert_history.json", "w", encoding="utf-8") as handle:
    json.dump(history, handle, indent=2)

for variant, payload in history["variants"].items():
    print(f"\n{variant}")
    print(pd.DataFrame(payload["epochs"])[
        ["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1", "eval_f1_negative", "eval_recall_negative"]
    ].to_string(index=False))
    print("train loss by epoch:")
    print(pd.DataFrame(payload["train_logs"])[["epoch", "loss"]].to_string(index=False))

## 14. Four-model comparison on val

Pulls the Phase 2 numbers from the uploaded `baseline_metrics.json` rather than
retyping them.

In [ ]:
with open("baseline_metrics.json", encoding="utf-8") as handle:
    baseline = json.load(handle)["metrics"]


def row_from_baseline(key, label):
    m = baseline[key]
    return {
        "model": label,
        "accuracy": m["accuracy"],
        "macro_F1": m["macro_f1"],
        "weighted_F1": m["weighted_f1"],
        "negative_F1": m["per_class"]["negative"]["f1-score"],
        "negative_recall": m["per_class"]["negative"]["recall"],
    }


def row_from_eval(final, label):
    return {
        "model": label,
        "accuracy": round(final["eval_accuracy"], 4),
        "macro_F1": round(final["eval_macro_f1"], 4),
        "weighted_F1": round(final["eval_weighted_f1"], 4),
        "negative_F1": round(final["eval_f1_negative"], 4),
        "negative_recall": round(final["eval_recall_negative"], 4),
    }


comparison = pd.DataFrame([
    row_from_baseline("dummy_most_frequent", "dummy"),
    row_from_baseline("tfidf_logreg_balanced", "tfidf_balanced"),
    row_from_eval(standard_final, "distilbert_standard"),
    row_from_eval(weighted_final, "distilbert_weighted"),
])

print("VAL SPLIT COMPARISON (n = {:,})\n".format(len(val_df)))
print(comparison.to_string(index=False))

best = comparison.loc[comparison["macro_F1"].idxmax()]
tfidf = comparison[comparison["model"] == "tfidf_balanced"].iloc[0]
print(f"\nbest macro F1: {best['model']} at {best['macro_F1']:.4f}")
print(f"vs TF-IDF baseline ({tfidf['macro_F1']:.4f}): {best['macro_F1'] - tfidf['macro_F1']:+.4f}")
if best["macro_F1"] <= tfidf["macro_F1"]:
    print("NOTE: the transformer did not beat the classical baseline on val.")

comparison.to_json("val_comparison.json", orient="records", indent=2)

## 15. Download the checkpoints and results

Produces `distilbert_standard.zip`, `distilbert_weighted.zip`,
`distilbert_history.json` and `val_comparison.json`.

Unzip the two model folders into `models/` in the local repo, and put the JSON
files in `results/`. Then run `python -m src.model.evaluate_test` locally.

In [ ]:
!zip -q -r distilbert_standard.zip distilbert_standard
!zip -q -r distilbert_weighted.zip distilbert_weighted
!ls -lh distilbert_*.zip

for filename in [
    "distilbert_standard.zip",
    "distilbert_weighted.zip",
    "distilbert_history.json",
    "val_comparison.json",
]:
    files.download(filename)